# Radar frame testbench

Interactive H5 reader to verify radar loading **frame by frame** (same processing as `visualize_sync_video.py`).

1. Edit **paths** below (or set `DATASET` to load from `datasets.json`).
2. Run **Open radar H5** once.
3. Scrub with the slider or edit `FRAME_IDX`.

| Panel | Source |
|-------|--------|
| Range-azimuth | `compute_range_azimuth` (dB, video left panel) |
| Radar BEV | `range_azimuth_to_bev` (`--show_radar_bev`) |

**Processing** (`RADAR_PROCESSING`): `legacy` (raw FFT+Capon), `ti` (chirp MTI + Hamming + zero-Doppler notch, then Capon), or `compare` (side-by-side).

**Index modes** (`INDEX_MODE`):
- `sync_idx` — `radar_idx` from sync (0 .. N_sync_blocks-1)
- `adc_frame` — sample-aligned ADC frame (0 .. max_adc_frame)
- `pair` — row in `sync_pairs.csv` (shows matched lidar_idx too) — **use for synced annotations**

**Annotations** (optional): set `ANNOTATIONS_PATH` and use `INDEX_MODE = "pair"`. Boxes are drawn on BEV panels in shared lateral/forward meters. See **Annotate** cell.

In [ ]:
from pathlib import Path

# --- Edit these, or set DATASET to auto-load from datasets.json ---
DATASET = "2026.05.10/18-05-08"  # e.g. "2026.05.10/18-05-08", or None

RADAR_H5 = "/Volumes/Seagate Portable Drive/data_collect_mobile/2026.05.10/radar/2026.05.10-21.28.12.h5"
CFG_FILE = "mmWaveStudio/mobile_collection/rng40_res0004_tx3_rx4.lua"
SYNC_SUMMARY = Path("res/2026.05.10/18-05-08/sync_summary.json")
SYNC_CSV = Path("res/2026.05.10/18-05-08/sync_pairs.csv")

# sync_idx | adc_frame | pair
INDEX_MODE = "sync_idx"

# legacy | ti | compare
RADAR_PROCESSING = "compare"

if DATASET:
    import sys

    _here = Path.cwd()
    _sync_dir = next((c for c in [_here, *_here.parents] if (c / "datasets.json").is_file()), None)
    if _sync_dir is None:
        _sync_dir = next((c / "sync" for c in [_here, *_here.parents] if (c / "sync" / "datasets.json").is_file()), _here)
    _src_dir = _sync_dir / "src"  # pipeline modules live in code/sync/src/
    if str(_src_dir) not in sys.path:
        sys.path.insert(0, str(_src_dir))
    from lib.dataset_config import load_dataset_paths

    paths = load_dataset_paths(DATASET)
    print(f"dataset: {DATASET}")
    RADAR_H5 = paths["radar_h5"]
    CFG_FILE = paths.get("cfg_file", CFG_FILE)
    SYNC_CSV = Path(paths.get("sync_csv", SYNC_CSV))
    SYNC_SUMMARY = Path(paths.get("sync_summary", SYNC_SUMMARY))

print("H5:", RADAR_H5)
print("Cfg:", CFG_FILE)
print("Sync CSV:", SYNC_CSV)
print("Sync summary:", SYNC_SUMMARY)
print("Index mode:", INDEX_MODE)

In [ ]:
import csv
import json
import sys
import time
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np

CODE_ROOT = Path.cwd()
if not (CODE_ROOT / "sync").is_dir() and (CODE_ROOT.parent / "sync").is_dir():
    CODE_ROOT = CODE_ROOT.parent
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
if str(CODE_ROOT / "sync") not in sys.path:
    sys.path.insert(0, str(CODE_ROOT / "sync"))

from lib.sync_annotations import (
    boxes_for_pair,
    build_sync_pair_maps,
    draw_topdown_boxes,
    load_annotations,
    pair_idx_from_radar,
    pick_box_two_clicks,
    save_annotations,
    upsert_box,
)
from lib.sync_utils import read_radar_packet_timestamps
from lib.bev_render import (
    SAMPLES_PER_H5_PACKET,
    compute_range_azimuth,
    effective_adc_packets_per_frame,
    infer_sync_packets_per_frame,
    load_radar_adc_frame,
    radar_adc_bounds,
    radar_adc_fingerprint,
    radar_samples_per_frame,
    radar_panel_axis,
    range_azimuth_to_bev,
    sync_radar_idx_to_adc_frame,
)
from utils.parse_config import radarConfig

print("Working dir:", CODE_ROOT)

In [ ]:
# --- Open radar H5 (run once) ---

def _resolve_cfg(path: str) -> str:
    candidates = [
        Path(path),
        CODE_ROOT / path,
        CODE_ROOT / "mmWaveStudio" / "server.lua",
    ]
    for p in candidates:
        if p.is_file():
            return str(p.resolve())
    raise FileNotFoundError(f"Radar cfg not found: {path}")


_cfg_path = _resolve_cfg(CFG_FILE)
_radar = radarConfig()
_radar.parse_radar(cfg_file=_cfg_path)

_packet_ts = read_radar_packet_timestamps(RADAR_H5)
N_PACKETS, SAMPLES_PER_FRAME, MAX_ADC_FRAME, TOTAL_SAMPLES = radar_adc_bounds(RADAR_H5, _radar)
ADC_PPF = effective_adc_packets_per_frame(_radar)

with open(SYNC_SUMMARY) as f:
    _summary = json.load(f)
SYNC_PPF = int(_summary.get("radar_packets_per_frame", 0))
if SYNC_PPF <= 0:
    SYNC_PPF = infer_sync_packets_per_frame(RADAR_H5, int(_summary.get("radar_frames", 0)))

SYNC_RADAR_IDX = []
SYNC_LIDAR_IDX = []
SYNC_DELTA_MS = []
if Path(SYNC_CSV).is_file():
    with open(SYNC_CSV) as f:
        for row in csv.DictReader(f):
            SYNC_RADAR_IDX.append(int(row["radar_idx"]))
            SYNC_LIDAR_IDX.append(int(row["lidar_idx"]))
            SYNC_DELTA_MS.append(float(row["delta_ms"]))

N_SYNC_BLOCKS = int(_summary.get("radar_frames", N_PACKETS // SYNC_PPF))

if INDEX_MODE == "adc_frame":
    SLIDER_MAX = int(MAX_ADC_FRAME)
elif INDEX_MODE == "pair":
    if not SYNC_RADAR_IDX:
        raise RuntimeError("pair mode needs sync_pairs.csv")
    SLIDER_MAX = len(SYNC_RADAR_IDX) - 1
else:
    SLIDER_MAX = max(0, N_SYNC_BLOCKS - 1)

print(f"Cfg: {_cfg_path}")
print(
    f"packets={N_PACKETS} | samples/frame={SAMPLES_PER_FRAME} "
    f"({_radar.num_adc_samples}x{_radar.num_chirps}x{_radar.num_rx}) | "
    f"ADC frames 0..{MAX_ADC_FRAME} | sync_ppf={SYNC_PPF} | adc_ppf={ADC_PPF}"
)
if SYNC_RADAR_IDX:
    print(
        f"sync_pairs: {len(SYNC_RADAR_IDX)} rows | "
        f"radar_idx {min(SYNC_RADAR_IDX)}..{max(SYNC_RADAR_IDX)}"
    )
print(f"Slider: INDEX_MODE={INDEX_MODE} -> 0..{SLIDER_MAX}")

In [ ]:
def _adc_frame_timestamp(adc_frame: int) -> float | None:
    if _packet_ts.size == 0:
        return None
    spf = radar_samples_per_frame(_radar)
    pi = (int(adc_frame) * spf) // SAMPLES_PER_H5_PACKET
    pi = int(np.clip(pi, 0, _packet_ts.size - 1))
    return float(_packet_ts[pi])


def resolve_adc_frame(frame_idx: int) -> tuple[int, dict]:
    """Map slider index -> adc_frame + metadata for title."""
    frame_idx = int(frame_idx)
    meta = {"index_mode": INDEX_MODE, "slider_idx": frame_idx}

    if INDEX_MODE == "adc_frame":
        if frame_idx < 0 or frame_idx > MAX_ADC_FRAME:
            raise IndexError(f"adc_frame {frame_idx} out of range [0, {MAX_ADC_FRAME}]")
        adc = frame_idx
        meta["sync_idx"] = None
        meta["pair_idx"] = None
    elif INDEX_MODE == "pair":
        if frame_idx < 0 or frame_idx >= len(SYNC_RADAR_IDX):
            raise IndexError(f"pair {frame_idx} out of range [0, {len(SYNC_RADAR_IDX)})")
        sync_idx = int(SYNC_RADAR_IDX[frame_idx])
        adc = sync_radar_idx_to_adc_frame(sync_idx, SYNC_PPF, _radar, MAX_ADC_FRAME)
        meta.update(
            {
                "sync_idx": sync_idx,
                "pair_idx": frame_idx,
                "lidar_idx": int(SYNC_LIDAR_IDX[frame_idx]),
                "delta_ms": float(SYNC_DELTA_MS[frame_idx]),
            }
        )
    else:
        if frame_idx < 0 or frame_idx > SLIDER_MAX:
            raise IndexError(f"sync_idx {frame_idx} out of range [0, {SLIDER_MAX}]")
        sync_idx = frame_idx
        adc = sync_radar_idx_to_adc_frame(sync_idx, SYNC_PPF, _radar, MAX_ADC_FRAME)
        meta["sync_idx"] = sync_idx
        meta["pair_idx"] = _radar_to_pair.get(sync_idx) if _radar_to_pair else None

    meta["adc_frame"] = int(adc)
    return int(adc), meta


def render_panels(frame_idx: int):
    adc_frame, meta = resolve_adc_frame(frame_idx)
    t0 = time.time()
    adc = load_radar_adc_frame(RADAR_H5, adc_frame, _radar)
    fp = radar_adc_fingerprint(adc)
    meta["load_s"] = time.time() - t0
    meta["timestamp_s"] = _adc_frame_timestamp(adc_frame)
    meta["radar_processing"] = RADAR_PROCESSING

    out = {"fp": fp, "meta": meta}
    modes = ("legacy", "ti") if RADAR_PROCESSING == "compare" else (RADAR_PROCESSING,)
    for mode in modes:
        range_az = compute_range_azimuth(adc, _radar, angle="Azimuth", processing=mode)
        suffix = f"_{mode}" if RADAR_PROCESSING == "compare" else ""
        out[f"range_az{suffix}"] = range_az
        out[f"bev{suffix}"] = range_azimuth_to_bev(range_az, _radar)
    return out


def _panel_vlim(img: np.ndarray) -> tuple[float, float] | None:
    v = img[np.isfinite(img)]
    if v.size == 0:
        return None
    nz = v[v > 0]
    if nz.size > 0:
        v = nz
    vmin = float(np.percentile(v, 2))
    vmax = float(np.percentile(v, 98))
    if vmax <= vmin:
        vmax = vmin + 1.0
    return vmin, vmax


def show_frame(frame_idx: int, *, compare_prev: bool = True):
    global _last_frame, _last_panels
    panels = render_panels(frame_idx)
    fp = panels["fp"]
    meta = panels["meta"]
    ts = meta.get("timestamp_s")
    ts_str = (
        datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        if ts is not None
        else "n/a"
    )

    if RADAR_PROCESSING == "compare":
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        views = [
            ("range_az_legacy", "Range-azimuth legacy"),
            ("range_az_ti", "Range-azimuth TI-style"),
            ("bev_legacy", "BEV legacy"),
            ("bev_ti", "BEV TI-style"),
        ]
        axes_flat = axes.ravel()
    else:
        fig, axes_flat = plt.subplots(1, 2, figsize=(12, 4.5))
        tag = RADAR_PROCESSING
        views = [
            ("range_az", f"Range-azimuth ({tag})"),
            ("bev", f"Radar BEV ({tag})"),
        ]
    for ax, (key, title) in zip(axes_flat, views):
        img = panels[key]
        vlim = _panel_vlim(img)
        use_bev = key.startswith("bev")
        axis = radar_panel_axis(img, _radar, use_bev=use_bev)
        kwargs = {
            "origin": axis["origin"],
            "aspect": axis["aspect"],
            "cmap": "viridis",
            "extent": axis["extent"],
        }
        if vlim is not None:
            kwargs["vmin"], kwargs["vmax"] = vlim
        im = ax.imshow(img, **kwargs)
        if axis["extent"] is not None:
            ax.set_xlim(axis["extent"][0], axis["extent"][1])
            ax.set_ylim(axis["extent"][2], axis["extent"][3])
        if use_bev and ann_boxes:
            draw_topdown_boxes(ax, ann_boxes)
        ax.set_title(title)
        ax.set_xlabel(axis["xlabel"])
        ax.set_ylabel(axis["ylabel"])
        plt.colorbar(im, ax=ax, fraction=0.046)

    extra = ""
    if meta.get("pair_idx") is not None:
        extra = (
            f"  pair={meta['pair_idx']}  lidar_idx={meta['lidar_idx']}  "
            f"delta={meta['delta_ms']:.1f}ms"
        )
    elif meta.get("sync_idx") is not None:
        extra = f"  sync_idx={meta['sync_idx']}"

    diff_line = ""
    if compare_prev and "_last_panels" in globals() and _last_frame is not None:
        for key, title in views:
            if key in panels and key in _last_panels:
                d = float(np.mean(np.abs(panels[key] - _last_panels[key])))
                diff_line += f"  mean|Δ{key}|={d:.3f}"

    fig.suptitle(
        f"{INDEX_MODE}={frame_idx}  proc={RADAR_PROCESSING}  adc_frame={meta['adc_frame']}  "
        f"cksum={fp['adc_checksum']}  abs_max={fp['adc_abs_max']:.0f}  "
        f"t={ts_str}  load={meta['load_s']:.2f}s{extra}{diff_line}",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

    _last_frame = int(frame_idx)
    _last_panels = panels
    return panels

In [ ]:
# --- Annotate: click two corners on radar BEV, save to ANNOTATIONS_PATH ---


def annotate_radar_pair(
    pair_idx: int,
    *,
    label: str = "object",
    box_id: str = "",
    color: str = "yellow",
    processing: str = "legacy",
):
    """Add one synced box for *pair_idx* (uses that pair's radar frame)."""
    global _annotations
    panels = render_panels(int(pair_idx))
    key = "bev" if RADAR_PROCESSING != "compare" else f"bev_{processing}"
    img = panels[key]
    axis = radar_panel_axis(img, _radar, use_bev=True)
    vlim = _panel_vlim(img)

    fig, ax = plt.subplots(figsize=(6, 5))
    kwargs = {
        "origin": axis["origin"],
        "aspect": axis["aspect"],
        "cmap": "viridis",
        "extent": axis["extent"],
    }
    if vlim is not None:
        kwargs["vmin"], kwargs["vmax"] = vlim
    ax.imshow(img, **kwargs)
    if axis["extent"] is not None:
        ax.set_xlim(axis["extent"][0], axis["extent"][1])
        ax.set_ylim(axis["extent"][2], axis["extent"][3])
    draw_topdown_boxes(ax, boxes_for_pair(_annotations, pair_idx))
    meta = panels["meta"]
    ax.set_title(
        f"Annotate pair={pair_idx}  lidar_idx={meta.get('lidar_idx')}  "
        f"adc_frame={meta['adc_frame']}"
    )
    ax.set_xlabel(axis["xlabel"])
    ax.set_ylabel(axis["ylabel"])
    plt.tight_layout()
    plt.show()

    box = pick_box_two_clicks(ax, label=label, box_id=box_id or f"pair{pair_idx}_{label}", color=color)
    if box is None:
        return None
    _annotations = upsert_box(_annotations, pair_idx, box)
    save_annotations(_annotations, ANNOTATIONS_PATH)
    print(f"Saved to {ANNOTATIONS_PATH}: pair {pair_idx} -> {box}")
    return box


# Example (INDEX_MODE must be "pair"): annotate_radar_pair(189, label="vehicle")

In [ ]:
# --- Manual frame index: edit and re-run this cell ---
FRAME_IDX = 0
show_frame(FRAME_IDX)

In [ ]:
# --- Interactive slider (requires ipywidgets) ---
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print("Install ipywidgets for slider: pip install ipywidgets")
else:
    start = 0
    if INDEX_MODE == "pair" and SYNC_RADAR_IDX:
        start = len(SYNC_RADAR_IDX) // 2
    elif INDEX_MODE == "sync_idx":
        start = min(SLIDER_MAX // 2, SLIDER_MAX)

    slider = widgets.IntSlider(
        value=min(start, SLIDER_MAX),
        min=0,
        max=int(SLIDER_MAX),
        step=1,
        description=INDEX_MODE,
        continuous_update=False,
        layout=widgets.Layout(width="80%"),
    )
    out = widgets.interactive_output(show_frame, {"frame_idx": slider})
    display(widgets.VBox([slider, out]))

In [ ]:
# --- Quick probe: first / mid / last index ---
probe = sorted({0, SLIDER_MAX // 2, SLIDER_MAX})
if INDEX_MODE == "pair" and SYNC_RADAR_IDX:
    probe = sorted({0, len(SYNC_RADAR_IDX) // 2, len(SYNC_RADAR_IDX) - 1})

rows = []
for idx in probe:
    adc, meta = resolve_adc_frame(idx)
    adc_data = load_radar_adc_frame(RADAR_H5, adc, _radar)
    fp = radar_adc_fingerprint(adc_data)
    rows.append((idx, meta.get("sync_idx"), adc, fp["adc_checksum"], meta.get("timestamp_s")))

print("slider | sync_idx | adc_frame | adc_checksum | timestamp_s")
for r in rows:
    print(f"{r[0]:6d} | {str(r[1]):8s} | {r[2]:9d} | {r[3]:12d} | {r[4]}")
if len({r[3] for r in rows}) < 2:
    print("WARN: identical checksums — check INDEX_MODE / sync_ppf")
else:
    print("OK: checksums vary across frames")